# Step 1.3 — Data Preprocessing

Preprocessing pipeline for the coronary centerline diffusion project, covering everything needed between the raw generated data (Steps 1.1 centerlines + 1.2 DRR projections) and a training-ready dataset: ID consistency, exploratory data analysis, train/val/test split, coordinate normalization, padding, image intensity normalization, per-case packaging, and the PyTorch Dataset/DataLoader.

Run from the repo root.

In [1]:
%cd "/Users/atefebahrami/Desktop/GT_Courses/2nd_Semster/CS6999/GitHub/cs6999-coronary-centerline-2"
%pwd

/Users/atefebahrami/Desktop/GT_Courses/2nd_Semster/CS6999/GitHub/cs6999-coronary-centerline-2


'/Users/atefebahrami/Desktop/GT_Courses/2nd_Semster/CS6999/GitHub/cs6999-coronary-centerline-2'

In [2]:
import json
from pathlib import Path

import numpy as np
import nibabel as nib

centerline_dir = Path("data/processed/centerlines")
drr_dir = Path("data/processed/DRR_Generation/Total")
raw_dir = Path("data/raw")
splits_dir = Path("data/splits")
splits_dir.mkdir(parents=True, exist_ok=True)

## 1. ID Consistency Check

**Why:** Step 1.1 (centerline extraction) and Step 1.2 (DRR generation) were run in separate sessions, weeks apart, on Kaggle and locally respectively. Before building anything on top of both, we need to confirm every case has *both* a centerline and a DRR projection set — a case missing one or the other would silently break packaging later if not caught now.

In [3]:
centerline_ids = {int(f.stem.split('_')[0]) for f in centerline_dir.glob("*_centerline.npy")}
drr_ids = {int(f.stem.split('_')[1]) for f in drr_dir.glob("case_*_projections.npz")}

print("Total centerline files:", len(centerline_ids))
print("Total DRR files:", len(drr_ids))
print("Centerline-only:", sorted(centerline_ids - drr_ids)[:10])
print("DRR-only:", sorted(drr_ids - centerline_ids)[:10])
print("Both match 1000?", len(centerline_ids) == len(drr_ids) == 1000)

Total centerline files: 1000
Total DRR files: 1000
Centerline-only: []
DRR-only: []
Both match 1000? True


**Observed:** 1000/1000 centerline files and 1000/1000 DRR files, with zero cases missing on either side (`Centerline-only: []`, `DRR-only: []`). Full ID consistency confirmed across the entire dataset.

## 2. Exploratory Data Analysis (EDA)

**Why:** Before designing padding, batching, or normalization, we need to know the actual shape of the data — how much centerlines vary in length across cases, what radius values look like, and whether the topology labels (endpoint / regular / bifurcation) are populated as expected across the whole dataset, not just a couple of spot-checked cases.

In [4]:
point_counts = []
radius_stats = []
for f in sorted(centerline_dir.glob("*_centerline.npy")):
    arr = np.load(f)
    point_counts.append(arr.shape[0])
    radius_stats.append((arr[:, 3].min(), arr[:, 3].max(), arr[:, 3].mean()))

point_counts = np.array(point_counts)
print(f"Point count -- min: {point_counts.min()}, max: {point_counts.max()}, "
      f"mean: {point_counts.mean():.0f}, median: {np.median(point_counts):.0f}")

radius_arr = np.array(radius_stats)
print(f"Radius -- overall min: {radius_arr[:,0].min():.2f}, max: {radius_arr[:,1].max():.2f}, "
      f"mean of means: {radius_arr[:,2].mean():.2f}")

Point count -- min: 351, max: 2884, mean: 1415, median: 1394
Radius -- overall min: 1.00, max: 10.30, mean of means: 3.73


In [5]:
label_counts = {0: 0, 1: 0, 2: 0}  # endpoint, regular, bifurcation
for f in sorted(centerline_dir.glob("*_centerline.npy")):
    arr = np.load(f)
    labels, counts = np.unique(arr[:, 4], return_counts=True)
    for l, c in zip(labels.astype(int), counts):
        label_counts[l] += c

print("Topology label totals across all 1000 cases:", label_counts)

Topology label totals across all 1000 cases: {0: 13823, 1: 1336220, 2: 65163}


**Observed:**
- **Point count:** min 351, max 2884, mean 1415, median 1394 -- a wide range (~8x between smallest and largest case). This directly tells us **padding/masking is required** before batching for training; centerlines cannot be stacked as-is.
- **Radius:** min 1.00mm, max 10.30mm, mean of per-case means 3.73mm -- clinically plausible for coronary vessels (main vessels typically 2-5mm, with the range covering smaller branches up to a widened case).
- **Topology labels:** endpoint 13,823 (~0.9%), regular 1,336,220 (~94.7%), bifurcation 65,163 (~4.4%). Proportions look reasonable at a glance, but ~65 "bifurcations" per case on average is far more than a realistic coronary tree's 5-15 true anatomical branch points. This indicates the skeleton-based topology labeling is picking up minor artifacts/noise along the skeleton, not exclusively genuine anatomical bifurcations. **Documented as a known limitation, not corrected** -- fixing it would require redoing centerline extraction across all 1000 cases, and the noisier signal is still learnable.

## 3. Train / Val / Test Split

**Why:** Case-level split (never by view or by point) to avoid data leakage -- every projection and every centerline point belonging to one case must stay entirely within one split. Split ratio (960/20/20) follows the training-set scale used by 3DGR-CAR (MICCAI 2024) on the same ImageCAS dataset. The split is derived from the **actual verified case IDs** confirmed in Step 1, not an assumed `range(1, 1001)`, since ImageCAS's case numbering is not perfectly contiguous.

In [6]:
def make_case_level_split(case_ids, val_frac=0.02, test_frac=0.02, seed=0):
    rng = np.random.default_rng(seed)
    ids = np.array(sorted(case_ids))
    rng.shuffle(ids)

    n_val = max(1, int(len(ids) * val_frac))
    n_test = max(1, int(len(ids) * test_frac))

    val_ids = ids[:n_val].tolist()
    test_ids = ids[n_val:n_val + n_test].tolist()
    train_ids = ids[n_val + n_test:].tolist()

    return {"train": train_ids, "val": val_ids, "test": test_ids}

splits = make_case_level_split(centerline_ids, seed=0)
print(f"train: {len(splits['train'])}, val: {len(splits['val'])}, test: {len(splits['test'])}")

with open(splits_dir / "case_splits.json", "w") as f:
    json.dump(splits, f, indent=2)
print("Saved to data/splits/case_splits.json")

train: 960, val: 20, test: 20
Saved to data/splits/case_splits.json


**Observed:** train: 960, val: 20, test: 20 -- matches the planned 960/20/20 split exactly.

## 4. Coordinate Normalization

**Why:** Saved centerline coordinates are raw voxel indices (confirmed below: max coordinate value 456, within typical CT array dimensions), not physical mm -- and different cases have different voxel grids/spacing, so raw indices aren't directly comparable across cases. The diffusion model needs coordinates on one consistent physical scale. Normalization statistics (mean/std) must be computed from the **training split only**, to avoid leaking val/test information into the transform applied to all splits -- this is why the split (Step 3) had to come before this step.

### 4.1 Confirm raw units and check voxel spacing

In [7]:
arr = np.load(centerline_dir / "1_centerline.npy")
print(arr[:5])
print("Max coordinate value:", arr[:, :3].max())

nii = nib.load(str(raw_dir / "1.label.nii.gz"))
print("Voxel spacing (mm):", nii.header.get_zooms()[:3])

[[110.         381.          61.           5.38516481   1.        ]
 [110.         381.          62.           4.89897949   1.        ]
 [110.         381.          63.           4.58257569   1.        ]
 [110.         381.          64.           4.47213595   1.        ]
 [110.         381.          65.           4.47213595   1.        ]]
Max coordinate value: 456.0
Voxel spacing (mm): (0.37695312, 0.37695312, 0.5)


**Observed:** max coordinate value 456 -- confirms raw voxel indices, not physical mm. Voxel spacing for case 1: (0.377, 0.377, 0.5) mm -- real, anisotropic spacing. Since ImageCAS pools scans from multiple scanners/protocols, spacing is expected to vary case to case, so every case's own spacing must be read individually rather than assuming one global value.

### 4.2 Convert every case's centerline from voxel indices to physical mm

In [8]:
def voxel_to_mm(centerline_arr, voxel_spacing):
    """Convert (x,y,z) voxel indices to physical mm. Radius (col 3) is
    also converted using the mean spacing as an isotropic approximation,
    since radius isn't a single-axis quantity. Topology label (col 4)
    passes through unchanged."""
    out = centerline_arr.copy()
    out[:, 0] *= voxel_spacing[0]
    out[:, 1] *= voxel_spacing[1]
    out[:, 2] *= voxel_spacing[2]
    out[:, 3] *= np.mean(voxel_spacing)
    return out

with open(splits_dir / "case_splits.json") as f:
    splits = json.load(f)

all_mm_centerlines = {}
missing_spacing = []

for case_id in splits["train"] + splits["val"] + splits["test"]:
    label_path = raw_dir / f"{case_id}.label.nii.gz"
    centerline_path = centerline_dir / f"{case_id}_centerline.npy"
    if not label_path.exists():
        missing_spacing.append(case_id)
        continue
    spacing = nib.load(str(label_path)).header.get_zooms()[:3]
    arr = np.load(centerline_path)
    all_mm_centerlines[case_id] = voxel_to_mm(arr, spacing)

print(f"Converted {len(all_mm_centerlines)} cases to mm; missing raw files: {len(missing_spacing)}")
if missing_spacing:
    print("Missing:", missing_spacing[:10])

Converted 1000 cases to mm; missing raw files: 0


**Observed:** Converted 1000 cases to mm; missing raw files: 0. Every case's raw NIfTI header was still available locally, so no cases had to be dropped at this step.

### 4.3 Compute normalization statistics from the training split only

In [9]:
train_points = np.concatenate(
    [all_mm_centerlines[cid][:, :4] for cid in splits["train"]],  # x,y,z,radius -- not topology label
    axis=0
)

coord_mean = train_points[:, :3].mean(axis=0)
coord_std = train_points[:, :3].std(axis=0)
radius_mean = train_points[:, 3].mean()
radius_std = train_points[:, 3].std()

print("Coordinate mean (x,y,z):", coord_mean)
print("Coordinate std (x,y,z):", coord_std)
print("Radius mean:", radius_mean, " Radius std:", radius_std)

norm_stats = {
    "coord_mean": coord_mean.tolist(),
    "coord_std": coord_std.tolist(),
    "radius_mean": float(radius_mean),
    "radius_std": float(radius_std),
}
with open(splits_dir / "normalization_stats.json", "w") as f:
    json.dump(norm_stats, f, indent=2)
print("Saved normalization_stats.json")

Coordinate mean (x,y,z): [ 94.08830834 100.57978384  66.54418308]
Coordinate std (x,y,z): [30.91405295 29.55421171 28.32170065]
Radius mean: 1.4954149819801614  Radius std: 0.4187858746555879
Saved normalization_stats.json


**Observed:** coordinate mean ~(94, 101, 67) mm, std ~(31, 30, 28) mm -- centered in the heart-region range with a consistent spread across axes. Radius mean 1.50mm, std 0.42mm. Statistics saved for reuse at inference/evaluation time to un-normalize model predictions.

### 4.4 Apply normalization to all cases, verify on the training split

In [10]:
def normalize_centerline(mm_arr, coord_mean, coord_std, radius_mean, radius_std):
    out = mm_arr.copy()
    out[:, :3] = (out[:, :3] - coord_mean) / coord_std
    out[:, 3] = (out[:, 3] - radius_mean) / radius_std
    # column 4 (topology label) untouched -- categorical, not continuous
    return out

normalized_centerlines = {
    cid: normalize_centerline(arr, coord_mean, coord_std, radius_mean, radius_std)
    for cid, arr in all_mm_centerlines.items()
}

sample_train = np.concatenate([normalized_centerlines[cid][:, :4] for cid in splits["train"]], axis=0)
print("Post-normalization train coord mean (should be ~0):", sample_train[:, :3].mean(axis=0))
print("Post-normalization train coord std (should be ~1):", sample_train[:, :3].std(axis=0))

Post-normalization train coord mean (should be ~0): [-9.74951887e-16  8.04899653e-15 -6.99314857e-16]
Post-normalization train coord std (should be ~1): [1. 1. 1.]


**Observed:** post-normalization training coordinate mean is effectively zero (~1e-15, floating-point rounding noise) on all three axes, and std is exactly 1.0 on all three axes. Confirms normalization is correctly computed and applied.

## 5. Padding

**Why:** Centerlines vary in length (351-2884 points, per Step 2's EDA), but training in batches requires fixed-size arrays. We pad every (already-normalized) centerline up to the dataset-wide maximum length, with a boolean mask marking which rows are real data vs. padding -- so the model/loss can ignore padded rows. Normalization is applied *before* padding, not after, so that padded zero-rows never contaminate the mean/std statistics.

In [11]:
MAX_LEN = 2884  # global max from EDA -- covers every case, no truncation

def pad_centerline(arr, max_len):
    """Pad a (N, 5) centerline array to (max_len, 5), with a boolean
    mask marking which rows are real data vs. padding."""
    n = arr.shape[0]
    if n > max_len:
        raise ValueError(f"Array length {n} exceeds max_len {max_len}")
    padded = np.zeros((max_len, arr.shape[1]), dtype=np.float32)
    mask = np.zeros(max_len, dtype=bool)
    padded[:n] = arr
    mask[:n] = True
    return padded, mask

padded_centerlines = {}
centerline_masks = {}
for cid, arr in normalized_centerlines.items():
    padded, mask = pad_centerline(arr, MAX_LEN)
    padded_centerlines[cid] = padded
    centerline_masks[cid] = mask

sample_id = splits["train"][0]
print(f"Case {sample_id}: original {normalized_centerlines[sample_id].shape[0]} points, "
      f"padded to {padded_centerlines[sample_id].shape[0]}, "
      f"mask sum {centerline_masks[sample_id].sum()} (should match original point count)")

Case 632: original 1393 points, padded to 2884, mask sum 1393 (should match original point count)


**Observed:** mask sum exactly matches the original point count (verified on a sample case, e.g. 1393 original points -> padded to 2884 -> mask sum 1393). Confirms padding preserves all real data and the mask correctly marks it.

## 6. Image Intensity Normalization

**Why:** DRR pixel values come from raw attenuation units with no natural bound, and need normalization before feeding into the model's CNN encoder. Same principle as coordinate normalization: compute statistics from the training split only, apply to all splits.

### 6.1 Check the actual intensity distribution first

In [12]:
sample_intensities = []
for cid in splits["train"][:50]:  # sample 50 cases -- enough for a reliable estimate
    d = np.load(drr_dir / f"case_{cid}_projections.npz", allow_pickle=True)
    sample_intensities.append(d["images"].flatten())

sample_intensities = np.concatenate(sample_intensities)
print("Min:", sample_intensities.min())
print("Max:", sample_intensities.max())
print("Mean:", sample_intensities.mean())
print("Std:", sample_intensities.std())
print("1st/99th percentile:", np.percentile(sample_intensities, [1, 99]))

Min: 0.0
Max: 5.6106424
Mean: 1.4719592
Std: 1.1716686
1st/99th percentile: [0.        3.9284711]


**Observed:** min 0.0, max 5.61, mean 1.47, std 1.17, 1st/99th percentile [0.0, 3.93]. The gap between the 99th percentile (3.93) and the true max (5.61) indicates a handful of extreme-value outlier pixels that would skew a plain min-max scale. Decision: use **percentile-clipped min-max normalization to [0,1]** (standard for image inputs to a CNN), rather than z-score standardization (more typical for non-image feature vectors like the centerline coordinates above).

### 6.2 Compute clip bounds from a larger training sample

In [13]:
sample_intensities = []
for cid in splits["train"][:200]:  # larger sample for more stable percentile estimates
    d = np.load(drr_dir / f"case_{cid}_projections.npz", allow_pickle=True)
    sample_intensities.append(d["images"].flatten())

sample_intensities = np.concatenate(sample_intensities)
p1, p99 = np.percentile(sample_intensities, [1, 99])
print(f"Clip bounds from 200 training cases -- p1: {p1:.4f}, p99: {p99:.4f}")

image_norm_stats = {"clip_min": float(p1), "clip_max": float(p99)}
with open(splits_dir / "image_norm_stats.json", "w") as f:
    json.dump(image_norm_stats, f, indent=2)
print("Saved image_norm_stats.json")

Clip bounds from 200 training cases -- p1: 0.0000, p99: 3.9058
Saved image_norm_stats.json


**Observed:** p1: 0.0000, p99: 3.9058 -- consistent with the smaller 50-case sample above, confirming the estimate is stable.

### 6.3 Normalization function and verification

In [14]:
def normalize_image(img, clip_min, clip_max):
    """Clip to [clip_min, clip_max] (from training data), then scale to [0,1]."""
    clipped = np.clip(img, clip_min, clip_max)
    return (clipped - clip_min) / (clip_max - clip_min)

sample_id = splits["train"][0]
d = np.load(drr_dir / f"case_{sample_id}_projections.npz", allow_pickle=True)
normalized_img = normalize_image(d["images"][0], p1, p99)
print(f"Case {sample_id} normalized image -- min: {normalized_img.min():.4f}, "
      f"max: {normalized_img.max():.4f}, mean: {normalized_img.mean():.4f}")

Case 632 normalized image -- min: 0.0000, max: 0.9672, mean: 0.2475


**Observed:** min 0.0000 (expected, since p1=0 means nothing clips at the bottom), max 0.9672 (below 1.0, since this particular case's brightest pixels don't reach the training-set-wide p99 bound -- expected and correct, not an error, since normalization is applied consistently relative to one shared reference scale across all cases, not rescaled per-case to fill [0,1] exactly).

## 7. Dataset Packaging

**Why:** All preprocessing pieces above (normalized/padded centerlines, normalized images, vessel masks, poses) exist as separate arrays computed on the fly. Before training, each case's full set of inputs needs to be combined into one file, so the training loop can load a case directly without recomputing normalization/padding every epoch. Packaged as **one file per case** (not one big stacked file per split), matching how the raw data is already stored and avoiding holding all 960 training cases in memory at once — this also plays naturally with a PyTorch `Dataset` that reads lazily per `__getitem__`.

In [15]:
centerline_dir = Path("data/processed/centerlines")
drr_dir = Path("data/processed/DRR_Generation/Total")
raw_dir = Path("data/raw")
packaged_dir = Path("data/processed/packaged")
packaged_dir.mkdir(parents=True, exist_ok=True)

with open("data/splits/case_splits.json") as f:
    splits = json.load(f)
with open("data/splits/normalization_stats.json") as f:
    norm_stats = json.load(f)
with open("data/splits/image_norm_stats.json") as f:
    img_norm_stats = json.load(f)

coord_mean = np.array(norm_stats["coord_mean"])
coord_std = np.array(norm_stats["coord_std"])
radius_mean = norm_stats["radius_mean"]
radius_std = norm_stats["radius_std"]
clip_min, clip_max = img_norm_stats["clip_min"], img_norm_stats["clip_max"]

MAX_LEN = 2884

def voxel_to_mm(arr, spacing):
    out = arr.copy()
    out[:, 0] *= spacing[0]; out[:, 1] *= spacing[1]; out[:, 2] *= spacing[2]
    out[:, 3] *= np.mean(spacing)
    return out

def normalize_centerline(mm_arr):
    out = mm_arr.copy()
    out[:, :3] = (out[:, :3] - coord_mean) / coord_std
    out[:, 3] = (out[:, 3] - radius_mean) / radius_std
    return out

def pad_centerline(arr, max_len):
    n = arr.shape[0]
    padded = np.zeros((max_len, arr.shape[1]), dtype=np.float32)
    mask = np.zeros(max_len, dtype=bool)
    padded[:n] = arr
    mask[:n] = True
    return padded, mask

def package_case(case_id, split_name):
    spacing = nib.load(str(raw_dir / f"{case_id}.label.nii.gz")).header.get_zooms()[:3]
    centerline_raw = np.load(centerline_dir / f"{case_id}_centerline.npy")
    centerline_mm = voxel_to_mm(centerline_raw, spacing)
    centerline_normed = normalize_centerline(centerline_mm)
    centerline_padded, centerline_mask = pad_centerline(centerline_normed, MAX_LEN)

    drr = np.load(drr_dir / f"case_{case_id}_projections.npz", allow_pickle=True)
    images_normed = np.clip(drr["images"], clip_min, clip_max)
    images_normed = (images_normed - clip_min) / (clip_max - clip_min)

    np.savez(
        packaged_dir / f"{case_id}.npz",
        centerline=centerline_padded,              # (MAX_LEN, 5) float32
        centerline_mask=centerline_mask,            # (MAX_LEN,) bool
        images=images_normed.astype(np.float32),    # (2, 512, 512), normalized [0,1]
        vessel_masks=drr["masks"].astype(np.float32),  # (2, 512, 512), raw
        poses=drr["poses"].astype(np.float32),       # (2, 3, 4)
        split=split_name,
    )

for split_name, ids in splits.items():
    for cid in ids:
        package_case(cid, split_name)
    print(f"{split_name}: packaged {len(ids)} cases")

print("Total packaged files:", len(list(packaged_dir.glob("*.npz"))))

train: packaged 960 cases
val: packaged 20 cases
test: packaged 20 cases
Total packaged files: 1000


In [16]:
d = np.load(packaged_dir / f"{splits['train'][0]}.npz", allow_pickle=True)
for k in d.files:
    print(k, d[k].shape if hasattr(d[k], "shape") else d[k])

centerline (2884, 5)
centerline_mask (2884,)
images (2, 512, 512)
vessel_masks (2, 512, 512)
poses (2, 3, 4)
split ()


**Observed:** train: packaged 960 cases, val: packaged 20 cases, test: packaged 20 cases -- total packaged files: 1000, matching the split exactly. Sample case fields: `centerline (2884, 5)`, `centerline_mask (2884,)`, `images (2, 512, 512)`, `vessel_masks (2, 512, 512)`, `poses (2, 3, 4)`, `split ()` -- all shapes correct (the empty tuple for `split` is just how `np.savez` stores a plain string, not an error; it reads back correctly as a string).

## 8. PyTorch Dataset & DataLoader

**Why:** The training loop needs to load packaged cases in batches, not one at a time by hand. A PyTorch `Dataset` wraps the per-case `.npz` loading logic; a `DataLoader` on top handles batching, shuffling, and (later) multiprocessing. This is the last step before the data is genuinely ready to feed into a model.

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader

class CoronaryCenterlineDataset(Dataset):
    def __init__(self, packaged_dir, case_ids):
        self.packaged_dir = Path(packaged_dir)
        self.case_ids = case_ids

    def __len__(self):
        return len(self.case_ids)

    def __getitem__(self, idx):
        case_id = self.case_ids[idx]
        d = np.load(self.packaged_dir / f"{case_id}.npz", allow_pickle=True)

        return {
            "centerline": torch.from_numpy(d["centerline"]).float(),           # (2884, 5)
            "centerline_mask": torch.from_numpy(d["centerline_mask"]).bool(),  # (2884,)
            "images": torch.from_numpy(d["images"]).float(),                  # (2, 512, 512)
            "vessel_masks": torch.from_numpy(d["vessel_masks"]).float(),      # (2, 512, 512)
            "poses": torch.from_numpy(d["poses"]).float(),                    # (2, 3, 4)
            "case_id": case_id,
        }


packaged_dir = "data/processed/packaged"

train_dataset = CoronaryCenterlineDataset(packaged_dir, splits["train"])
val_dataset = CoronaryCenterlineDataset(packaged_dir, splits["val"])
test_dataset = CoronaryCenterlineDataset(packaged_dir, splits["test"])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

print(f"train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

train: 960, val: 20, test: 20


In [18]:
batch = next(iter(train_loader))
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v))

centerline torch.Size([8, 2884, 5]) torch.float32
centerline_mask torch.Size([8, 2884]) torch.bool
images torch.Size([8, 2, 512, 512]) torch.float32
vessel_masks torch.Size([8, 2, 512, 512]) torch.float32
poses torch.Size([8, 2, 3, 4]) torch.float32
case_id torch.Size([8]) torch.int64


**Observed:** train: 960, val: 20, test: 20 -- matches the split. One sampled batch (batch_size=8): `centerline (8, 2884, 5) float32`, `centerline_mask (8, 2884) bool`, `images (8, 2, 512, 512) float32`, `vessel_masks (8, 2, 512, 512) float32`, `poses (8, 2, 3, 4) float32`, `case_id (8,) int64`. All shapes and dtypes correct -- batching works as expected.

Notes: `num_workers=0` used for now on the M4 (multiprocessing workers can be finicky with `.npz` loading; worth testing `num_workers=2-4` later once training is underway). `batch_size=8` is a starting value, not yet tuned against actual training memory usage.

## Summary

All preprocessing steps complete and verified:

| Step | Status | Key output |
|---|---|---|
| 1. ID consistency check | Done | 1000/1000 matched |
| 2. EDA | Done | point counts, radius, topology distributions |
| 3. Train/val/test split | Done | `data/splits/case_splits.json` (960/20/20) |
| 4. Coordinate normalization | Done | `data/splits/normalization_stats.json` |
| 5. Padding | Done | MAX_LEN=2884, boolean mask per case |
| 6. Image intensity normalization | Done | `data/splits/image_norm_stats.json` |
| 7. Dataset packaging | Done | `data/processed/packaged/` (1000 files) |
| 8. PyTorch Dataset & DataLoader | Done | verified correct batch shapes/dtypes |

**Step 1.3 (Data Preprocessing) is fully complete.** Combined with Steps 1.1 (centerline extraction) and 1.2 (DRR generation), all of Step 1 (Data Preparation) is done.

**Next step:** Step 2 -- 3D Centerline Reconstruction Model (2.1 deterministic baseline, 2.2 diffusion model architecture).